# DIT Natural Language Processing lesson 2026

## Bidirectional recurrent neural networks

In this notebook, we build a recurrent neural network (BiRNN). RNNs consider both the outputs at time $t-1$ from the left and from the right.

In [ ]:
! pip install gensim

In [ ]:
# Importing the dependencies
import glob
import numpy as np
import os
import pandas as pd
import tarfile

from gensim.models.keyedvectors import KeyedVectors

from keras.models import Sequential
from keras.layers import Bidirectional          # <- this is the new wrapper
from keras.layers import Dense, Dropout, Flatten, Input, SimpleRNN

from nltk.tokenize import TreebankWordTokenizer
from random import shuffle
from tqdm.auto import tqdm
from urllib import request

In [ ]:
PATH_TO_GOOGLENEWS_VECTORS ="https://www.dropbox.com/s/965dir4dje0hfi4/GoogleNews-vectors-negative300.bin.gz?dl=1"
GOOGLE_VECTORS = "GoogleNews-vectors-negative300.bin.gz"

def download_file(url_to_file, path_to_file):
  if os.path.isfile(path_to_file):
    print("A local copy of the file exists already:", path_to_file, "\nDoing nothing")
  else:
    request.urlretrieve(url_to_file, path_to_file)

def download_corpus():
    """
    A function to download and shuffle the data (through huggingface)

    The dataset is at https://huggingface.co/datasets/stanfordnlp/imdb
    """

    splits = {'train': 'plain_text/train-00000-of-00001.parquet',
              'test': 'plain_text/test-00000-of-00001.parquet',
              'unsupervised': 'plain_text/unsupervised-00000-of-00001.parquet'}
    df = pd.read_parquet("hf://datasets/stanfordnlp/imdb/" + splits["train"])

    # shuffle the dataset
    df = df.sample(frac=1).reset_index(drop=True)
    return df

In [ ]:
# Downloading and loading the embeddings
download_file(PATH_TO_GOOGLENEWS_VECTORS, GOOGLE_VECTORS)
word_vectors = KeyedVectors.load_word2vec_format(GOOGLE_VECTORS,
    binary=True, limit=400000)

# Downloading and loading the corpus
dataset = download_corpus()
dataset

In [ ]:
# Tokenizing and vectorizing all the instances
def tokenize_and_vectorize(dataset):
  tokenizer = TreebankWordTokenizer()
  vectorized_data = []
  # expcted = [] this line appears in the book, but it's not necessary here!
  for sample in tqdm(dataset['text']):
    # print(sample)
    tokens = tokenizer.tokenize(sample)
    sample_vecs = []
    for token in tokens:
      try:
        sample_vecs.append(word_vectors[token])
      except KeyError:
        pass # No matching token in the Google w2v vocab
    np_vec = np.vstack(sample_vecs)
    vectorized_data.append(np_vec)
  return vectorized_data

In [ ]:
# Extracting the expected output for all the instances
def collect_expected(dataset):
    """Peel off the target values (classes) from the dataset """
    expected = []
    for sample in dataset['label']:
        expected.append(sample)
    return expected

In [ ]:
# Vectorising the dataset and extracting the gold standard
x = tokenize_and_vectorize(dataset)
y = collect_expected(dataset)

In [ ]:
# Define training and validation data (even if it's called test here)
split_point = int(len(x)*.8)

x_train = x[:split_point] # there's a typo in this line, if copying from the book
y_train = y[:split_point]
# This is actually a validation partition. I keep the names from the book
x_test = x[split_point:]
y_test = y[split_point:]

In [ ]:
# Input parameters
maxlen = 400
embedding_dims = 300

# Network parameters
batch_size = 32
epochs = 2
num_neurons = 50

In [ ]:
# Padding and truncating the input is not strictly necessary for RNNs; we
# due to the recursion strategy we are applying here and to have a fair
# comparison against the CNN (same input length)
class Collator:
  def __init__(self,
                maxlen,
                batch_size,
                ) -> None:
    self.maxlen = maxlen
    self.batch_size = batch_size

  def padding_and_truncating(self, x, y):
    """
    Add zeros at the end of the representation for short instances,
    truncate longer ones to the maxlen
    """
    vec_dim = len(x[0][0])
    N = len(x)
    X = np.zeros((N, self.maxlen, vec_dim))  # preallocate a np array
    Y = np.array(y)

    for i, tokens in enumerate(x):
      length = min(len(tokens), self.maxlen)
      if length > 0:
        X[i, :length] = np.asarray(tokens[:length])  # fill the np array
    return X, Y

  def collate(self, X, Y, N, epochs = 1):
    """
    This method is used to feed batches into the `model.fit` method
    """
    for _ in range(epochs):
      """
      This `for _ in range(epochs):` loop is here because
      the `for` below needs to be able to be called `epochs` times
      so each time the `for` is called the iterator is replenished
      for the new epoch
      """
      for start in range(0, N, self.batch_size):
        end = start + self.batch_size
        x_batch, y_batch = self.padding_and_truncating(
          X[start:end],
          Y[start:end]
          )
        yield x_batch, y_batch

In [ ]:
collator = Collator(maxlen = maxlen, batch_size = batch_size)

In [ ]:
model = Sequential()
# Adding one bidirectional recurrent layer
model.add(Input([maxlen, embedding_dims]))
model.add(Bidirectional(SimpleRNN(
    num_neurons,
    return_sequences=True),
    )
    )

# Compare it against what we had last time
# model.add(Input([maxlen, embedding_dims]))
# model.add(SimpleRNN(
#     num_neurons,
#     return_sequences=True)
#     )

In [ ]:
# Adding a dropout layer (remember why?)
model.add(Dropout(.2))

# Adding a flattening layer
model.add(Flatten())

# Adding the classifier
model.add(Dense(1, activation='sigmoid'))

In [ ]:
# Compiling the network
model.compile('rmsprop',
              'binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
# Training the network

model.fit(
    collator.collate(x_train, y_train, N = len(x_train), epochs=epochs),
    steps_per_epoch=len(x_train) // batch_size,
    validation_data=collator.collate(x_test, y_test, N = len(x_test), epochs=epochs),
    validation_steps=len(x_test) // batch_size,
    epochs=epochs
    )


In [ ]:
# Notice we have both positive and negative words here
sample_negative = """I hate that the dismal weather had me down for so long,
when will it break! Ugh, when does happiness return? The sun is
blinding and the puffy clouds are too thin. I can't wait for the weekend."""
# Super positive sample
sample_positive = """I love that incredible weather!
I feel like this happiness will last forever!
The sun is super nice and the breeze is soothing.
I could stay like this forever.
"""

# The first value is a "fake" class (this is the expected input)
new_instances = [
    {"text": sample_negative, "label": 0},
    {"text": sample_positive, "label": 1}
    ]
data_dummy = pd.DataFrame.from_records(new_instances)

x_dummy = tokenize_and_vectorize(data_dummy)
y_dummy = collect_expected(data_dummy)

x_dummy_padded, y_dummy = collator.padding_and_truncating(x_dummy, y_dummy)
preds = model.predict(x_dummy_padded)
print(preds)

In [ ]:
# Get the class by thresholding the logits
(preds > 0.5).astype("int32")

**End of the notebook**